# Monthly-reset swing — prototype

A call swing whose strike resets every delivery month from the model's own
projection of that month's own price, observed one month ahead — not a strike
fixed at inception.

**No real term sheet exists for this.** Built as a generic prototype, per
explicit direction: *"just strike price is changing month ahead, i.e. not
fixed."* Every simplification is named in
[`docs/DESIGN-MONTHLY-RESET-SWING-2026-09-13.md`](docs/DESIGN-MONTHLY-RESET-SWING-2026-09-13.md)
sec.2.1/sec.14 rather than silently assumed — call-only, one point observation
per month (not a delivery-period average), a model-internal index (not a
real published one), a deal-wide volume limit (not a monthly one), and no
partially-fixed months. Read that document before using a number from this
notebook for anything beyond exploring the mechanism.

Sections 1-4 price the point-reset (Release 1A); section 5 prices the averaged
reset (Release 1B, a restricted prototype of that design's own broader
specification -- see sec.14.1 and the 2026-09-14 independent review's R-06
finding) on a smaller term sheet. Both were found to have a real
fixing-window defect (R-01/R-02, same review) and were fixed 2026-09-14;
every number below reflects the corrected code.

In [ ]:
import os, warnings

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import reset_terms as rt
import reset_forward as rf
import reset_swing_exact as rse
import reset_swing_averaged as rsa

pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (12, 3.4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
warnings.filterwarnings("ignore", category=FutureWarning)

_missing = [n for n in ("ResetSwingTerms", "build_reset_schedule")
           if not hasattr(rt, n)]
_missing += [n for n in ("value_point_reset_call_swing", "compute_deltas")
            if not hasattr(rse, n)]
_missing += [n for n in ("value_averaged_reset_call_swing", "compute_deltas")
            if not hasattr(rsa, n)]
if _missing:
    raise RuntimeError(
        "stale kernel — missing " + ", ".join(_missing) +
        " (Kernel > Restart Kernel and Run All).")

print("ready")


## 1. Term sheet — a prototype convention, not a real deal

In [ ]:
TERMS = rt.ResetSwingTerms(
    val_date="2026-01-01",
    storage_start="2026-03-01",     # a full calendar month lead from val_date --
    storage_end="2026-08-31",       # this prototype has no historical-fixing input
    daily_max_mwh=1_000.0,          # maximum daily exercised volume
    v_step_mwh=500.0,               # the DP's clip size
    global_min_mwh=0.0,             # 0 = exercise is optional up to the max
    global_max_mwh=60_000.0,        # deal-wide, not per-month (sec.14's own simplification)
    vol=0.5, sMR=1.0,               # illustrative, not calibrated -- see docs/STATUS.md
    discount_rate=0.05,
    n_p=15,                         # price-tree half-width; see the convergence note below
)
SCHEDULE = rt.build_reset_schedule(TERMS)

print(f"{len(SCHEDULE.months)} delivery months, {TERMS.storage_start:%Y-%m-%d} .. "
     f"{TERMS.storage_end:%Y-%m-%d}")
_rows = [{"month": m.label, "fixing date": m.fixing_date.date(),
         "exercise window": f"{m.exercise_dates[0].date()} .. {m.exercise_dates[-1].date()}",
         "days": len(m.exercise_dates)} for m in SCHEDULE.months]
display(pd.DataFrame(_rows))


## 2. The forward curve

Any daily curve `Storage`/`run_valuation` already accepts works here — a flat
placeholder is used so this notebook runs without a market data file. The
month-ahead index this contract resets from is the model's own conditional
projection of the SAME curve, not a second, independent data source (sec.14's
"model-internal index" simplification) — a shaped curve here changes the
*strike* it resets to as well as the *price* it exercises against, which is
exactly the mechanism the delta section below decomposes.

In [ ]:
DAYS = pd.date_range("2020-01-01", "2030-12-31", freq="D")
CURVE = pd.Series(25.0, index=DAYS)
# A summer/winter shape, so the reset actually moves between months.
CURVE.loc["2026-04-01":"2026-05-31"] = 22.0
CURVE.loc["2026-06-01":"2026-08-31"] = 30.0

_window = CURVE.loc[TERMS.storage_start:TERMS.storage_end]
fig, ax = plt.subplots()
ax.step(_window.index, _window.values, where="post", color="black", lw=1.4)
ax.set_ylabel("EUR/MWh"); ax.set_title("Input curve over the deal window")
plt.tight_layout(); plt.show()


## 3. Price it

In [ ]:
PV = rse.value_point_reset_call_swing(TERMS, SCHEDULE, daily_curve=CURVE)
print(f"PV = {PV:,.2f} EUR")

_lattice = rf.build_lattice(TERMS.val_date, TERMS.storage_start, TERMS.storage_end,
                            vol=TERMS.vol, sMR=TERMS.sMR, n_p=TERMS.n_p,
                            daily_curve=CURVE, discount_rate=TERMS.discount_rate)
_quotes = rf.project_month_end_quotes(_lattice, SCHEDULE.month_end_dates)
_date_span = _lattice["date_span"]
print("\nReset strike each month resets to (the model's own root-date view; the realised")
print("strike will differ once each fixing actually happens):")
for _m, _end in zip(SCHEDULE.months, SCHEDULE.month_end_dates):
    _u = _date_span.get_loc(_end)
    _fix_idx = _date_span.get_loc(_m.fixing_date)
    _k0 = float(_quotes[_u][0, TERMS.n_p])
    print(f"  {_m.label}: fixes {_m.fixing_date.date()}, root-date projected K = {_k0:.3f} EUR/MWh")


## 4. Hedge sensitivities

Three measures (sec.9.3): the **total** curve delta (what actually matters —
a real curve move changes both legs together), and two diagnostics that
freeze one leg to isolate the other. `total` is authoritative; the two legs
are an attribution, not independently tradeable instruments, and are not
asserted to sum to `total` exactly (only to first order in the bump).

In [ ]:
DELTAS = rse.compute_deltas(TERMS, SCHEDULE, CURVE, bump_eur_mwh=0.10)
print(f"total delta        {DELTAS['total']:>12,.2f}  EUR per EUR/MWh (parallel shift)")
print(f"physical-leg delta {DELTAS['physical_leg']:>12,.2f}  (spot bumped, strike frozen)")
print(f"index-leg delta    {DELTAS['index_leg']:>12,.2f}  (strike bumped, spot frozen)")
print(f"sum of legs        {DELTAS['physical_leg']+DELTAS['index_leg']:>12,.2f}  vs total above")
print()
print("A small total next to two large, mostly-offsetting legs is the point of indexing")
print("the strike at all: this contract is deliberately insensitive to a parallel curve")
print("move. What it is NOT insensitive to is BASIS risk -- physical and index prices")
print("moving differently from each other -- which this parallel-shift delta does not")
print("measure at all. Do not read a small total delta as a small-risk contract.")


## 5. The averaged reset (Release 1B)

Everything above uses a single point observation to fix each month's strike (Release 1A).
The generic prototype also supports an **averaged** reset: the strike is the equal-weighted
average of the model's own month-ahead projection, taken on *every* calendar day of the
preceding month, not read off a single fixing date.

A smaller, separate term sheet is used below (2 delivery months instead of 6, `n_p=8` instead
of 15) purely so this section runs quickly -- the averaged reset's own state space adds a
genuinely continuous running-average axis (`r`), discretised and linearly interpolated, which
the point-reset benchmark above does not need at all (sec.6.4: a point reset's strike is an
exact function of a single lattice node, not a continuous quantity). Picking `n_r`/`r_lo`/`r_hi`
matters here in a way it never does for point-reset -- see the note in the pricing cell below.

In [ ]:
TERMS_AVG = rt.ResetSwingTerms(
    val_date="2026-01-01",
    storage_start="2026-05-01", storage_end="2026-06-30",   # spans the curve's own May -> June step
    daily_max_mwh=1_000.0, v_step_mwh=1_000.0,
    global_min_mwh=0.0, global_max_mwh=8_000.0,
    vol=0.4, sMR=1.0, discount_rate=0.05,
    n_p=8,
)
SCHEDULE_AVG = rt.build_reset_schedule(TERMS_AVG)

_rows = [{"month": m.label, "fixing date": m.fixing_date.date(),
         "exercise window": f"{m.exercise_dates[0].date()} .. {m.exercise_dates[-1].date()}",
         "days": len(m.exercise_dates)} for m in SCHEDULE_AVG.months]
display(pd.DataFrame(_rows))


### Price it, and compare to the point reset on the same (smaller) term sheet

In [ ]:
N_R, R_LO, R_HI = 150, 10.0, 45.0   # accumulator grid: must bracket EVERY delivery month's
# own projected-quote range, not just the curve's spot level -- a shaped curve gives each
# month a genuinely different range, and value_averaged_reset_call_swing raises rather than
# silently clamping if the grid turns out too narrow (found the hard way -- see
# reset_swing_averaged.py's own module docstring). Convergence in n_r is first-order,
# O(1/n_r), because the value-vs-strike surface has a genuine kink at the exercise boundary --
# tests/test_reset_swing_averaged.py pins that rate as a regression, not just an observation.

PV_POINT_SMALL = rse.value_point_reset_call_swing(TERMS_AVG, SCHEDULE_AVG, daily_curve=CURVE)
PV_AVERAGED = rsa.value_averaged_reset_call_swing(
    TERMS_AVG, SCHEDULE_AVG, CURVE, n_r=N_R, r_lo=R_LO, r_hi=R_HI)

print(f"point-reset PV    = {PV_POINT_SMALL:,.2f} EUR  (same term sheet, single-observation strike)")
print(f"averaged-reset PV = {PV_AVERAGED:,.2f} EUR  (equal-weighted month-ahead average)")
print(f"difference        = {PV_AVERAGED - PV_POINT_SMALL:,.2f} EUR "
     f"({100 * (PV_AVERAGED / PV_POINT_SMALL - 1):.1f}%)")


### Hedge sensitivities, same three measures as section 4

In [ ]:
DELTAS_AVG = rsa.compute_deltas(TERMS_AVG, SCHEDULE_AVG, CURVE, n_r=N_R, r_lo=R_LO, r_hi=R_HI,
                               bump_eur_mwh=0.10)
print(f"total delta        {DELTAS_AVG['total']:>12,.2f}  EUR per EUR/MWh (parallel shift)")
print(f"physical-leg delta {DELTAS_AVG['physical_leg']:>12,.2f}  (spot bumped, strike frozen)")
print(f"index-leg delta    {DELTAS_AVG['index_leg']:>12,.2f}  (strike bumped, spot frozen)")
print(f"sum of legs        {DELTAS_AVG['physical_leg'] + DELTAS_AVG['index_leg']:>12,.2f}  vs total above")
print()
print("Same caveat as section 4: a small total next to two large, offsetting legs is not a")
print("small-risk contract -- it says nothing about basis risk between the physical and index")
print("prices, which a parallel-shift delta cannot see by construction.")


## 6. What this notebook does and does not show

- **Does**: price a generic (no real term sheet) month-ahead-reset call swing exactly, at
  real volatility, for both reset conventions this prototype supports -- a single-observation
  point reset (Release 1A, sec.3/4) with a verified state-space reduction (the strike axis
  collapses to the fixing node exactly, sec.6.4), and an equal-weighted month-ahead average
  (Release 1B, sec.5) with a genuinely continuous, interpolated running-average state -- and
  three-way hedge attribution for both.
- **Does not**: use a real market curve or a real published index (the index is always this
  model's own projection of the same curve, never a second data source); per-month volume
  limits or partially-fixed months (both sec.14 simplifications); a literal brute-force
  cross-check beyond two delivery months for the averaged reset (the multi-month CHAINING
  mechanism itself is brute-force verified against every non-anticipating policy on a tiny
  scenario tree -- see `tests/test_reset_swing_averaged.py` -- a longer chain is not expected
  to behave differently, just to cost more to enumerate literally). See
  `docs/DESIGN-MONTHLY-RESET-SWING-2026-09-13.md` sec.13/sec.14 for exactly what is and is
  not built, and `docs/STATUS.md` for the current test count and what each test actually
  pins.